# Check of one O4b noise realization

Measured ASD of H1 and L1 around one injection time (compared with the O5c design curve) and
whitened strain of the analysis segment: if PSD and data are consistent, the whitened strain
follows $N(0, 1)$.

Requires the O4b frames (see `O4B_DATA_DIR` in `config/params.py`) and
`segments/injection_times.json` (written by `make_segments.py`).
Run from `bilby_inference/scripts/`.

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

SCRIPT_DIR = os.getcwd()                     # .../bilby_inference/scripts
BILBY_DIR = os.path.dirname(SCRIPT_DIR)
sys.path.insert(0, BILBY_DIR)
sys.path.insert(0, os.path.dirname(BILBY_DIR))

import bilby
bilby.core.utils.log.setup_logger(log_level="WARNING")
from inference_utils import make_ifo_network_real, make_ifo_network, make_waveform_generator, F_LOW

with open(os.path.join(BILBY_DIR, "segments", "injection_times.json")) as f:
    injection_times = json.load(f)["geocent_times"]
t_c = injection_times[0] # choose a realization HERE
print(f"{len(injection_times)} injection times; using t_c = {t_c}")

In [ ]:
# Interferometers with the O5 PSD and with the PSD measured from O4b data

ifos = make_ifo_network_real(t_c)
ifos_O5 = make_ifo_network(make_waveform_generator().frequency_array)

for ifo in ifos:
    print(f"{ifo.name}: start {ifo.strain_data.start_time}, duration {ifo.duration} s, "
          f"fs {ifo.sampling_frequency} Hz, band [{ifo.minimum_frequency}, {ifo.maximum_frequency}] Hz")

In [ ]:
# Measured ASD of H1 and L1 (from O4b data) and O5 design ASD

fig, ax = plt.subplots(figsize=(8, 5))
for ifo in ifos:
    ax.loglog(ifo.frequency_array, ifo.amplitude_spectral_density_array, lw=0.8, label=f"{ifo.name} O4b (measured)")
f_O5 = ifos_O5[0].frequency_array
m = f_O5 >= F_LOW
ax.loglog(f_O5[m], ifos_O5[0].amplitude_spectral_density_array[m], "k--", lw=1.2, label="O5 design")
ax.set_xlim(F_LOW, ifos[0].maximum_frequency)
ax.set_ylim(1e-24, 1e-20)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel(r"$\sqrt{S_n (f)}$ [1/$\sqrt{\rm Hz}$]")
ax.set_title(f"t_c = {t_c:.0f}")
ax.legend(frameon=False)
plt.show()

In [ ]:
# If PSD and data are consistent, the whitened strain is N(0, 1).
# Isolated spikes above ~5σ indicate a glitch: in that case discard this time and pick another one.

fig, axes = plt.subplots(2, 2, figsize=(12, 6), gridspec_kw=dict(width_ratios=[3, 1]))
for row, ifo in zip(axes, ifos):
    w = ifo.whitened_time_domain_strain
    t = ifo.time_array - t_c
    edge = int(1.0 * ifo.sampling_frequency)          # drop 1 s at each end (Tukey window)
    w_in, t_in = w[edge:-edge], t[edge:-edge]
    row[0].plot(t_in, w_in, lw=0.3)
    row[0].axhline(5, color="r", ls=":", lw=0.8); row[0].axhline(-5, color="r", ls=":", lw=0.8)
    row[0].set_ylabel(f"{ifo.name} whitened")
    x = np.linspace(-5, 5, 200)
    row[1].hist(w_in, bins=100, density=True, histtype="step")
    row[1].plot(x, norm.pdf(x), "k--", lw=1)
    row[1].set_yscale("log"); row[1].set_ylim(1e-5, 1)
    print(f"{ifo.name}: std = {np.std(w_in):.3f}, max |w| = {np.max(np.abs(w_in)):.2f}")
axes[-1, 0].set_xlabel("t - t_c [s]")
plt.tight_layout()
plt.show()